In [1]:
# ============================================================
# Graph Representation Learning
# ============================================================

import os
import pickle
import warnings

import numpy as np
import pandas as pd

import networkx as nx

from node2vec import Node2Vec

from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
GRAPH_PATH = "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/logistics_graph.pkl"

CENTRALITY_PATH = "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/node_centrality_features.csv"

COMMUNITY_PATH = "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/community_features.csv"

FEATURE_STORE_PATH = "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/graph_feature_store.csv"

In [8]:
with open(GRAPH_PATH, "rb") as f:
    G = pickle.load(f)

centrality_df = pd.read_csv(CENTRALITY_PATH)

community_df = pd.read_csv(COMMUNITY_PATH)

graph_feature_store = pd.read_csv(FEATURE_STORE_PATH)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1657
Edges: 2783


In [11]:
#config

EMBEDDING_DIM = 32

WALK_LENGTH = 20

NUM_WALKS = 200

WINDOW = 10

MIN_COUNT = 1

WORKERS = 4


In [12]:
node2vec = Node2Vec(
    G,
    dimensions=EMBEDDING_DIM,
    walk_length=WALK_LENGTH,
    num_walks=NUM_WALKS,
    workers=WORKERS,
    seed=42
)

model = node2vec.fit(
    window=WINDOW,
    min_count=MIN_COUNT,
    batch_words=256
)

Computing transition probabilities:   0%|          | 0/1657 [00:00<?, ?it/s]

In [13]:
embeddings = []

for node in G.nodes():

    vector = model.wv[str(node)]

    row = [node] + vector.tolist()

    embeddings.append(row)

In [14]:
columns = (
    ["facility"] +
    [f"embedding_{i}" for i in range(1, EMBEDDING_DIM + 1)]
)

embedding_df = pd.DataFrame(
    embeddings,
    columns=columns
)

embedding_df.head()

,facility,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5,embedding_6,embedding_7,embedding_8,embedding_9,...,embedding_23,embedding_24,embedding_25,embedding_26,embedding_27,embedding_28,embedding_29,embedding_30,embedding_31,embedding_32
0,IND000000AAL,0.263162,0.042453,-0.829589,0.457591,0.213730,0.275269,0.282538,0.366559,-0.226895,...,0.787881,-0.743866,-0.031620,0.178212,-0.291145,0.358351,-0.539173,-0.737894,-0.212151,-0.090376
1,IND411033AAA,0.285970,0.144569,-0.694518,0.333120,0.241145,0.280766,0.288131,0.184986,-0.178370,...,0.677057,-0.588522,-0.040627,0.144676,-0.344052,0.384066,-0.420182,-0.657973,-0.195128,-0.189729
2,IND000000AAQ,0.799222,-0.123339,0.062078,-0.245318,0.858890,-0.380566,-0.486448,0.977739,-0.275752,...,0.838439,0.350900,-0.051388,0.293958,-0.096071,1.138553,-0.456282,0.821317,0.123347,0.026336
3,IND700028AAB,0.761594,-0.373733,-0.165127,-0.238841,0.674781,-0.104967,-0.225300,0.971849,-0.213452,...,0.735642,-0.124512,-0.187825,0.408717,0.004318,0.839459,0.046399,0.911205,-0.129513,0.143897
4,IND000000AAS,1.011091,-0.617957,0.862963,-0.130742,1.093615,-0.273499,-0.299010,0.468011,-0.227805,...,0.948252,0.456410,-0.990524,1.074426,-0.416523,0.855354,0.813491,0.515749,0.291677,-1.397603


In [15]:
OUTPUT_PATH = "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/graph_node_embeddings.csv"

embedding_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)

Saved: /content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/graph_node_embeddings.csv


vector mag check

In [16]:
embedding_matrix = embedding_df.drop(
    columns=["facility"]
).values

norms = np.linalg.norm(
    embedding_matrix,
    axis=1
)

pd.Series(norms).describe()

,0
count,1657.000000
mean,4.060009
std,0.899420
min,1.843785
25%,3.396406
50%,3.992805
75%,4.641383
max,7.195377


In [17]:
def nearest_facilities(
        facility,
        top_k=10
):

    vec = model.wv[str(facility)]

    sims = []

    for other in G.nodes():

        if other == facility:
            continue

        score = cosine_similarity(
            vec.reshape(1,-1),
            model.wv[str(other)].reshape(1,-1)
        )[0][0]

        sims.append((other, score))

    sims.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return pd.DataFrame(
        sims[:top_k],
        columns=["facility","similarity"]
    )

In [18]:
nearest_facilities(
    "IND000000ACB"
)

,facility,similarity
0,IND201301AAM,0.991461
1,IND110037AAK,0.990983
2,IND110043AAC,0.987025
3,IND110095AAB,0.982401
4,IND110052AAA,0.981677
5,IND000000AEL,0.977972
6,IND121002AAB,0.977891
7,IND302022AAD,0.973789
8,IND110030AAD,0.949379
9,IND127201AAA,0.927793


In [19]:
node_features = graph_feature_store.copy()

In [20]:
community_map = dict(
    zip(
        node_features["facility"],
        node_features["community_id"]
    )
)

pagerank_map = dict(
    zip(
        node_features["facility"],
        node_features["pagerank"]
    )
)

betweenness_map = dict(
    zip(
        node_features["facility"],
        node_features["betweenness_centrality"]
    )
)

hub_map = dict(
    zip(
        node_features["facility"],
        node_features["hub_score"]
    )
)

authority_map = dict(
    zip(
        node_features["facility"],
        node_features["authority_score"]
    )
)

In [21]:
edge_rows = []

for u, v in G.edges():

    same_community = int(
        community_map[u] == community_map[v]
    )

    pr_diff = abs(
        pagerank_map[u]
        - pagerank_map[v]
    )

    bt_diff = abs(
        betweenness_map[u]
        - betweenness_map[v]
    )

    hub_diff = abs(
        hub_map[u]
        - hub_map[v]
    )

    auth_diff = abs(
        authority_map[u]
        - authority_map[v]
    )

    sim = cosine_similarity(
        model.wv[str(u)].reshape(1,-1),
        model.wv[str(v)].reshape(1,-1)
    )[0][0]

    edge_rows.append([
        u,
        v,
        same_community,
        pr_diff,
        bt_diff,
        hub_diff,
        auth_diff,
        sim
    ])

In [22]:
edge_feature_df = pd.DataFrame(
    edge_rows,
    columns=[
        "source",
        "destination",

        "same_community",

        "pagerank_difference",
        "betweenness_difference",

        "hub_score_difference",
        "authority_score_difference",

        "embedding_cosine_similarity"
    ]
)

In [23]:
EDGE_OUTPUT = (
    "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/"
    "graph_edge_features.csv"
)

edge_feature_df.to_csv(
    EDGE_OUTPUT,
    index=False
)

print("Saved:", EDGE_OUTPUT)

Saved: /content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/graph_edge_features.csv


In [24]:
graph_ml_feature_store = (
    graph_feature_store
    .merge(
        embedding_df,
        on="facility",
        how="left"
    )
)

In [25]:
graph_ml_feature_store.shape

(1657, 43)

In [27]:
GRAPH_ML_PATH = (
    "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/"
    "graph_ml_feature_store.csv"
)

graph_ml_feature_store.to_csv(
    GRAPH_ML_PATH,
    index=False
)

print("Saved:", GRAPH_ML_PATH)

Saved: /content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/graph_ml_feature_store.csv


In [28]:
for hub in [
    "IND000000ACB",
    "IND562132AAA",
    "IND501359AAE",
    "IND421302AAG"
]:
    print("\n")
    print(hub)
    print(nearest_facilities(hub,5))



IND000000ACB
       facility  similarity
0  IND201301AAM    0.991461
1  IND110037AAK    0.990983
2  IND110043AAC    0.987025
3  IND110095AAB    0.982401
4  IND110052AAA    0.981677


IND562132AAA
       facility  similarity
0  IND560083AAB    0.980331
1  IND560058AAD    0.977817
2  IND560085AAA    0.967366
3  IND560060AAB    0.944878
4  IND560078AAC    0.944606


IND501359AAE
       facility  similarity
0  IND500039AAD    0.992850
1  IND500010AAB    0.984629
2  IND500080AAB    0.982067
3  IND501141AAA    0.947082
4  IND501101AAB    0.936789


IND421302AAG
       facility  similarity
0  IND410401AAA    0.984647
1  IND400072AAA    0.979961
2  IND400102AAB    0.978846
3  IND400701AAB    0.975714
4  IND421401AAC    0.975614
